In [ ]:
import numpy as np
import pandas as pd
import os
import torch
import matplotlib.pyplot as plt
import pydicom
from PIL import Image
from tqdm import tqdm
import pickle

DATA_PATH = "D:/ML/RSNA2024"

In [ ]:
df = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
df = df.fillna("Normal/Mild")


dfDescr = pd.read_csv(os.path.join(DATA_PATH, "train_series_descriptions.csv"))
uniqueSeriesDescr = np.array(['Sagittal T2/STIR', 'Sagittal T1', 'Axial T2'])
dfDescr.set_index("study_id", inplace=True)
uniqueStudIds = dfDescr.index.unique()

dfCoord = pd.read_csv(os.path.join(DATA_PATH, "train_label_coordinates.csv"))

In [ ]:
dfDescr.head()

In [ ]:
from DicomDataset import *

### Position

The x, y, and z coordinates of the upper left hand corner (center of the first voxel transmitted) of the image, in mm

### Orientation

This means that the dicom world (or patient) coordinate system is LPS:

X is Right to Left (RL)
Y is Anterior to Posterior (AP)
Z is Inferior to Superior (IS)

The Image Orientation (Patient) dicom tag is (0020,0037), and is defined as 6 elements: "Ax, Ay, Az, Bx, By, Bz".

When described as a 3x3 rotation matrix R, it's equivalent to:


$$
\left(\begin{array}{cc} 
A_x & B_x & 0\\
A_y & B_y & 0\\
A_z & B_z & 0\\
\end{array}\right)
$$ 

### Pixel Spacing

Physical distance in the patient between the center of each pixel, specified by a numeric pair - adjacent row spacing (delimiter) adjacent column spacing in mm.

-----

## Data Extraction Process

1. For all patients create PatientData
1. Get all sagittal scans
1. Process every slice with the vertebrae detector
1. For every level L
    1. Get all patches from bounding boxes @ level L (extend the boxes to get the spinal canal and foramina!)
    1. Use a random slice where all levels can be seen to extract the axial slices @ level L. Also center crop them.

--> For every patient the data consists of separate data for every level

```json
{
    "L1/L2":
        {"sagittalPatches": [...], "axialSlices":[...]},
    "L2/L3":
        {"sagittalPatches": [...], "axialSlices":[...]},
    "L3/L4":
        ...
}

```

In [ ]:
startXPos=300
endXPos=550
test.scans[0].slices[6].plot([startXPos,endXPos])
worldPosStart = test.scans[0].slices[6].getWorldPosition(0,startXPos)
worldPosEnd = test.scans[0].slices[6].getWorldPosition(0,endXPos)
print(worldPosStart)
print(worldPosEnd)

axScan = test.getAxialScans()[0]
foundSlices = test.getSlicesInRangeDirection(axScan, worldPosEnd, worldPosStart, Direction.Z)
plt.figure(figsize=(10,10))
numImages = int(np.ceil(np.sqrt(len(foundSlices))))
for i,s in enumerate(foundSlices):
    plt.subplot(numImages,numImages,i+1)
    plt.imshow(s.data)
    plt.axis("off")

In [ ]:
if os.path.exists(os.path.join(DATA_PATH,"./rawData.pkl")):
    with open(os.path.join(DATA_PATH,"./rawData.pkl"), "rb") as f:
        allData = pickle.load(f)
else:
    allData = {}
    for studyId in tqdm(uniqueStudIds):
        seriesIds = dfDescr.loc[studyId]["series_id"].to_numpy()
        seriesDescriptions = dfDescr.loc[studyId]["series_description"].to_numpy()
        # print(seriesDescriptions)
        scanMapping = []
        for seriesIndex,serId in enumerate(seriesIds):
            seriesDescr = seriesDescriptions[seriesIndex]
            if "Sagittal" in seriesDescr:
                orient = OrientationType.Sagittal
            elif "Axial" in seriesDescr or "Transversal" in seriesDescr:
                orient = OrientationType.Axial
            elif "Frontal" in seriesDescr:
                orient = OrientationType.Frontal
            else:
                orient = OrientationType.Unknown
            folder = os.path.join(DATA_PATH, f"train_images/{studyId}/{serId}")
            scanMapping.append((orient, folder))

        # test = PatientData(scanMapping)
        allData[studyId] = PatientData(scanMapping)
            
    with open(os.path.join(DATA_PATH,"./rawData.pkl"), "wb") as f:
        pickle.dump(allData, f)
